# Dekodowanie krok po kroku

Notebook jest samodzielny: funkcje sa zaimplementowane ponizej w komorkach.
Nie importuje lokalnych modulow projektu i nie uruchamia plikow .py.
Uruchamiaj komorki od gory. Wgraj dane na Dysk i ustaw katalog roboczy:
```python
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/colab
```


In [ ]:
%pip install numpy scipy soundfile samplerate matplotlib pandas h5py


## Importy

In [ ]:
import argparse
import json
import os
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
import samplerate
from scipy.signal import chirp, correlate, fftconvolve
from scipy.stats import linregress, theilslopes
from pathlib import Path
from types import SimpleNamespace
import uuid


## 1. Konfiguracja


In [ ]:
ZESTAW = '1sesja'
data_dir = Path('dane') / ZESTAW
recording = data_dir / ('2sesja_2zrodla.wav' if ZESTAW == '2sesja_2spk' else 'nagranie.wav')
symulacja = ZESTAW == 'symulacja'
args = SimpleNamespace(
    bundle_dir=str(data_dir),
    recording=str(recording),
    ir_len=8192 if symulacja else 96000,
    pre_roll=256,
    excitation_gain=0.8500000238418579 if ZESTAW == '1sesja' else None,
    no_drift=symulacja,
    all_methods=not symulacja,
    out_dir=str(Path('wyniki') / ZESTAW / str(uuid.uuid4())),
    plot=True,
    fmin_plot=60.0 if ZESTAW == '1sesja' else 100.0 if symulacja else 1500.0,
)


## 2. Sekwencje pobudzające


In [ ]:
def golay_pair(n: int):
    A, B = (np.array([1.0]), np.array([1.0]))
    for _ in range(n):
        A, B = (np.concatenate([A, B]), np.concatenate([A, -B]))
    return (A, B)
MLS_TAPS = {8: [8, 6, 5, 4], 9: [9, 5], 10: [10, 7], 12: [12, 6, 4, 1], 14: [14, 5, 3, 1], 16: [16, 14, 13, 11], 17: [17, 14], 18: [18, 11]}
GOLD_PAIRS = {5: ([5, 3], [5, 4, 3, 2]), 6: ([6, 5], [6, 5, 2, 1]), 7: ([7, 3], [7, 3, 2, 1]), 8: ([8, 6, 5, 4], [8, 7, 6, 1]), 9: ([9, 4], [9, 6, 4, 3]), 10: ([10, 7], [10, 9, 8, 6]), 12: ([12, 6, 4, 1], [12, 11, 10, 4]), 16: ([16, 14, 13, 11], [16, 15, 13, 4])}

def generate_mls(taps, m):
    reg = [1] * m
    seq = []
    for _ in range(2 ** m - 1):
        seq.append(1.0 if reg[-1] else -1.0)
        fb = 0
        for t in taps:
            fb ^= reg[t - 1]
        reg = [fb] + reg[:-1]
    return np.array(seq)

def reciprocal_mls(seq):
    if len(seq) <= 1:
        return np.array(seq, copy=True)
    return np.concatenate([seq[:1], seq[:0:-1]])

def gold_family(m, n_codes):
    t1, t2 = GOLD_PAIRS[m]
    mls1 = generate_mls(t1, m)
    mls2 = generate_mls(t2, m)
    codes = [mls1, mls2]
    for k in range(len(mls1)):
        if len(codes) >= n_codes:
            break
        codes.append(mls1 * np.roll(mls2, k))
    return codes[:n_codes]

def hadamard_matrix(n):
    H = np.array([[1.0]])
    while H.shape[0] < n:
        H = np.block([[H, H], [H, -H]])
    return H

def next_pow2(n):
    return int(np.power(2, np.ceil(np.log2(max(1, n)))))


## 3. Chirp, sinusy i zapis WAV


In [ ]:
def sine_tone(fs, dur_s, freq, amp=0.8):
    n = int(round(dur_s * fs))
    t = np.arange(n, dtype=float) / fs
    sig = amp * np.sin(2.0 * np.pi * freq * t)
    fade = int(min(n // 20, fs * 0.01))
    if fade > 0:
        ramp = np.linspace(0.0, 1.0, fade)
        sig[:fade] *= ramp
        sig[-fade:] *= ramp[::-1]
    return sig

def write_wav_multichannel(path, fs, channels, peak=0.85):
    data = np.column_stack([np.asarray(ch, dtype=np.float32) for ch in channels])
    mx = float(np.max(np.abs(data))) if data.size else 1.0
    if mx > 0:
        data *= peak / mx
    os.makedirs(os.path.dirname(path) if os.path.dirname(path) else '.', exist_ok=True)
    sf.write(path, data, fs, subtype='FLOAT')
    return float(peak / mx) if mx > 0 else 1.0


## 4. Dekodery i tabela metod


In [ ]:
FMAX_HZ = 20000.0
DRIFT_QUALITY = 'sinc_best'
OKNO_ZGRUBNE_S = 2.0
OKNO_DOKLADNE = 2
PROG_POMIARU = 0.1
PROG_RESZTY = 0.02
PROG_KWADRAT = 1.0

def bundle_mls(m):
    return generate_mls(MLS_TAPS[m], m)

def xcorr(y, ref, ir_len):
    c = fftconvolve(y, ref[::-1], mode='full')
    start = len(ref) - 1
    return c[start:start + ir_len]

def fragment(y, seg, L, ir_len, pre):
    return y[seg['start'] - pre:seg['start'] + L + ir_len - 1]

def decode_solo_golay(y, meta, ir_len, pre=0):
    A, B = golay_pair(meta['golay_n'])
    L = len(A)
    seg = {s['name']: s for s in meta['segments']}
    irs = []
    for j in range(meta['speakers']):
        ha = xcorr(fragment(y, seg[f'solo_golay_spk{j + 1}_A'], L, ir_len, pre), A, ir_len + pre)
        hb = xcorr(fragment(y, seg[f'solo_golay_spk{j + 1}_B'], L, ir_len, pre), B, ir_len + pre)
        irs.append((ha + hb) / (2.0 * L))
    return irs

def decode_solo_mls(y, meta, ir_len, pre=0):
    mls = bundle_mls(meta['mls_m'])
    L = len(mls)
    seg = {s['name']: s for s in meta['segments']}
    return [xcorr(fragment(y, seg[f'solo_mls_spk{j + 1}'], L, ir_len, pre), mls, ir_len + pre) / L for j in range(meta['speakers'])]

def decode_golay_hadamard(y, meta, ir_len, pre=0):
    A, B = golay_pair(meta['golay_n'])
    L = len(A)
    k = meta['hadamard_size']
    seg = {s['name']: s for s in meta['segments']}
    rows = []
    for i in range(k):
        ha = xcorr(fragment(y, seg[f'golay_had_row{i + 1}_A'], L, ir_len, pre), A, ir_len + pre)
        hb = xcorr(fragment(y, seg[f'golay_had_row{i + 1}_B'], L, ir_len, pre), B, ir_len + pre)
        rows.append((ha + hb) / (2.0 * L))
    decoded = hadamard_matrix(k).T @ np.array(rows)
    return list(decoded[:meta['speakers']] / k)

def decode_mls_hadamard(y, meta, ir_len, pre=0):
    mls = bundle_mls(meta['mls_m'])
    L = len(mls)
    k = meta['hadamard_size']
    seg = {s['name']: s for s in meta['segments']}
    rows = [xcorr(fragment(y, seg[f'mls_had_row{i + 1}'], L, ir_len, pre), mls, ir_len + pre) / L for i in range(k)]
    decoded = hadamard_matrix(k).T @ np.array(rows)
    return list(decoded[:meta['speakers']] / k)

def decode_mls_reciprocal(y, meta, ir_len, pre=0):
    mls = bundle_mls(meta['mls_m'])
    L = len(mls)
    seg = next((s for s in meta['segments'] if s['name'] == 'mls_reciprocal_pair'))
    yy = fragment(y, seg, L, ir_len, pre)
    return [xcorr(yy, code, ir_len + pre) / L for code in (mls, reciprocal_mls(mls))]

def decode_gold(y, meta, ir_len, pre=0):
    codes = gold_family(meta['gold_m'], meta['speakers'])
    L = len(codes[0])
    seg = next((s for s in meta['segments'] if s['name'] == 'gold_all'))
    yy = fragment(y, seg, L, ir_len, pre)
    return [xcorr(yy, code, ir_len + pre) / L for code in codes]
METHODS = {'solo_golay': ('Solo Golay', decode_solo_golay, '#1f77b4'), 'golay_hadamard': ('Golay+Hadamard', decode_golay_hadamard, '#2ca02c'), 'solo_mls': ('Solo MLS', decode_solo_mls, '#ff7f0e'), 'mls_hadamard': ('MLS+Hadamard', decode_mls_hadamard, '#d62728'), 'mls_reciprocal': ('MLS reciprocal', decode_mls_reciprocal, '#17becf'), 'gold': ('Gold', decode_gold, '#9467bd')}
PLOT_PAIRS = [('golay', ['solo_golay', 'golay_hadamard']), ('mls', ['solo_mls', 'mls_hadamard']), ('mls_reciprocal', ['solo_mls', 'mls_reciprocal']), ('gold', ['gold']), ('gold_vs_solo', ['gold', 'solo_mls', 'solo_golay'])]


## 5. Wczytywanie danych


In [ ]:
def read_wav_mono(path):
    data, fs = sf.read(path, always_2d=True)
    return (fs, np.ascontiguousarray(data[:, 0], dtype=np.float64))

def excitation_scale(bundle_dir, meta, override):
    if override is not None:
        return override
    if 'excitation_gain' in meta:
        return meta['excitation_gain']
    wav = os.path.join(bundle_dir, 'bundle.wav')
    if os.path.exists(wav):
        return float(np.max(np.abs(sf.read(wav)[0])))
    return None


## 6. Synchronizacja i korekcja dryftu


In [ ]:
def linear_chirp(fs, dur_s, f0=600.0, f1=6000.0, amp=0.6):
    n = int(round(dur_s * fs))
    t = np.arange(n, dtype=float) / fs
    sig = amp * chirp(t, f0=f0, t1=dur_s, f1=f1, method='linear', phi=-90)
    fade = int(min(n // 10, fs * 0.005))
    ramp = np.linspace(0.0, 1.0, fade)
    sig[:fade] *= ramp
    sig[-fade:] *= ramp[::-1]
    return sig

def pozycja_chirpu(y, wzorzec, srodek, okno):
    lo = max(0, srodek - okno)
    hi = min(len(y), srodek + okno + len(wzorzec))
    c = np.abs(correlate(y[lo:hi], wzorzec, mode='valid', method='fft'))
    i = int(np.argmax(c))
    poz = float(lo + i)
    if 0 < i < c.size - 1:
        mianownik = c[i - 1] - 2.0 * c[i] + c[i + 1]
        if mianownik != 0.0:
            poz += 0.5 * (c[i - 1] - c[i + 1]) / mianownik
    return (poz, i in (0, c.size - 1))

def zmierz_dryft(y, meta, fs):
    syncs = [s for s in meta['segments'] if s['kind'] == 'sync']
    wzorzec = linear_chirp(fs, (syncs[0]['end'] - syncs[0]['start']) / fs)
    t = np.array([s['start'] / fs for s in syncs])
    okno = int(round(OKNO_ZGRUBNE_S * fs))
    zgrubne, brzegi = ([], [])
    for s in syncs:
        poz, brzeg = pozycja_chirpu(y, wzorzec, s['start'], okno)
        zgrubne.append(poz - s['start'])
        brzegi.append(brzeg)
    zgrubne = np.array(zgrubne)
    a, b, _, _ = theilslopes(zgrubne, t)
    dokladne = []
    for k, s in enumerate(syncs):
        poz, brzeg = pozycja_chirpu(y, wzorzec, s['start'] + int(round(a * t[k] + b)), OKNO_DOKLADNE)
        dokladne.append(poz - s['start'])
        brzegi.append(brzeg)
    p = linregress(t, zgrubne)
    reszty = np.abs(zgrubne - (p.slope * t + p.intercept))
    czas = float(np.ptp(t))
    kwadrat = abs(float(np.polyfit((t - t[0]) / czas, zgrubne - zgrubne[0], 2)[0]))
    if any(brzegi):
        raise ValueError('Chirp synchronizacji na brzegu okna wyszukiwania')
    if np.any(np.abs(zgrubne - np.array(dokladne)) > PROG_POMIARU):
        raise ValueError('Rozbiezne polozenie chirpu w szerokim i waskim oknie')
    if reszty.max() > max(PROG_POMIARU, PROG_RESZTY * float(np.ptp(zgrubne))) or kwadrat > PROG_KWADRAT:
        raise ValueError('Przesuniecia chirpow nie narastaja liniowo')
    return (p.slope / fs, zgrubne)

def korekcja_dryftu(y, meta, fs):
    eps, _ = zmierz_dryft(y, meta, fs)
    y = samplerate.resample(y, 1.0 / (1.0 + eps), DRIFT_QUALITY)
    eps2, przesuniecia = zmierz_dryft(y, meta, fs)
    return (y, int(round(float(przesuniecia[0]))), {'eps': eps, 'residual_eps': eps2})


## 7. Charakterystyki częstotliwościowe


In [ ]:
def poczatki_okien(solo_golay, zero, fs):
    starts = {}
    for j, ir in enumerate(solo_golay, 1):
        peak = int(np.argmax(np.abs(ir[:zero + int(round(0.005 * fs))])))
        starts[j] = peak - int(round(0.001 * fs))
    return starts

def compute_fft_mag_db(ir, fs, fmax=None):
    mag_db = 20.0 * np.log10(np.maximum(np.abs(np.fft.rfft(ir)), 1e-12))
    f = np.fft.rfftfreq(len(ir), d=1.0 / fs)
    if fmax is not None:
        m = f <= float(fmax)
        f, mag_db = (f[m], mag_db[m])
    return (f, mag_db)

def plot_methods_per_speaker(keys, irs, title, out_path, fs, fmin):
    n_spk = len(irs[keys[0]])
    fig, axes = plt.subplots(n_spk, 1, figsize=(14, 3 + 3 * n_spk), squeeze=False, sharex=True)
    fig.suptitle(title, fontsize=14)
    axes = [row[0] for row in axes]
    for j, ax in enumerate(axes):
        for key in keys:
            label, _, color = METHODS[key]
            f, mag_db = compute_fft_mag_db(irs[key][j], fs)
            visible = (f >= fmin) & (f <= min(FMAX_HZ, fs / 2))
            ax.plot(f[visible], mag_db[visible], color=color, lw=0.7, label=label)
        ax.set_ylabel(f'spk{j + 1}  |H(f)| [dB]')
    for ax in axes:
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8, loc='lower left')
    axes[-1].set_xlabel('f [Hz]')
    axes[-1].set_xlim(fmin, FMAX_HZ)
    plt.tight_layout()
    plt.savefig(out_path, dpi=160)
    plt.close()


## 8. Manifest


In [ ]:
bundle_dir = os.path.abspath(args.bundle_dir)
with open(os.path.join(bundle_dir, 'manifest.json'), encoding='utf-8') as f:
    meta = json.load(f)
fs = meta['fs']


## 9. Nagranie i synchronizacja


In [ ]:
_, y = read_wav_mono(args.recording)
gain = excitation_scale(bundle_dir, meta, args.excitation_gain)
offset, drift = (0, None)
if not args.no_drift:
    y, offset, drift = korekcja_dryftu(y, meta, fs)
shifted = dict(meta, segments=[dict(s, start=s['start'] + offset, end=s['end'] + offset) for s in meta['segments']])


## 10. Dekodowanie


In [ ]:
names = {s['name'] for s in meta['segments']}
decoded = {}
for key, (label, decode, _) in METHODS.items():
    if not args.all_methods and key not in ('solo_golay', 'golay_hadamard'):
        continue
    if key == 'mls_reciprocal' and 'mls_reciprocal_pair' not in names:
        continue
    print(f'{label}...')
    decoded[key] = [ir / (gain or 1.0) for ir in decode(y, shifted, args.ir_len, args.pre_roll)]


## 11. Zapis IR i metadanych


In [ ]:
out_dir = os.path.abspath(args.out_dir or os.path.join(bundle_dir, 'decoded'))
os.makedirs(out_dir, exist_ok=True)
files = []
for key, irs in decoded.items():
    for j, ir in enumerate(irs, 1):
        files.append(f'decoded_{key}_spk{j}.npy')
        np.save(os.path.join(out_dir, files[-1]), ir)
run = {'fs': fs, 'recording': os.path.relpath(args.recording).replace(os.sep, '/'), 'bundle_manifest': meta, 'sync_mode': 'none' if args.no_drift else 'full', 'offset_samples': offset, 'drift': drift, 'zero_index': args.pre_roll, 'ir_len_after_zero': args.ir_len, 'excitation_gain': gain, 'files': files}
with open(os.path.join(out_dir, 'decode_run.json'), 'w', encoding='utf-8') as f:
    json.dump(run, f, indent=2, ensure_ascii=False)
print(f'Wyniki: {out_dir}')


## 12. Wykresy FFT


In [ ]:
if args.plot:
    n = int(round(0.1 * fs))
    starts = poczatki_okien(decoded['solo_golay'], args.pre_roll, fs)
    windows = {key: [ir[starts[j]:starts[j] + n] for j, ir in enumerate(irs, 1)] for key, irs in decoded.items()}
    n_spk = meta['speakers']
    for suffix, keys in PLOT_PAIRS:
        if all((k in windows for k in keys)):
            title = ' / '.join((METHODS[k][0] for k in keys)) + ' — okno analizy: 100 ms'
            plot_methods_per_speaker(keys, windows, title, os.path.join(out_dir, f'fft_{n_spk}spk_{suffix}.png'), fs, args.fmin_plot)
    plot_methods_per_speaker(list(windows), windows, 'Wszystkie metody — okno analizy: 100 ms', os.path.join(out_dir, 'fft_all_methods.png'), fs, args.fmin_plot)


In [ ]:
Path('wybrany_przebieg.json').write_text(json.dumps({'decoded_dir': str(Path(out_dir).resolve())}))
print(out_dir)
